# S01E02 — Find Him

**Zadanie:** Namierzyć, która z podejrzanych osób (z S01E01) przebywała blisko jednej z elektrowni atomowych.  
Następnie ustalić jej poziom dostępu i kod elektrowni, i wysłać dane do `/verify`.

**Techniki:** Function Calling, Haversine, wielokrokowy agent

**Dane wejściowe z S01E01:** lista podejrzanych (imię, nazwisko, rok urodzenia)

In [ ]:
import os, json, math, requests
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY     = os.getenv("AI_DEVS_API_KEY")
OPENAI_KEY  = os.getenv("OPENAI_API_KEY")
client      = OpenAI(api_key=OPENAI_KEY)
VERIFY_URL  = "https://hub.ag3nts.org/verify"
HUB_BASE    = "https://hub.ag3nts.org"
print("API_KEY:", API_KEY[:6] + "..." if API_KEY else "BRAK")

In [ ]:
# --- Podejrzani z zadania S01E01 ---
# Uzupełnij listą osób, które wysłałeś do Hub-u jako podejrzanych w S01E01.
# Format: {"name": "Jan", "surname": "Kowalski", "birthYear": 1985}

SUSPECTS = [
    # Przykład — zastąp prawdziwymi danymi z S01E01:
    # {"name": "Anna",   "surname": "Nowak",     "birthYear": 1987},
    # {"name": "Tomasz", "surname": "Wiśniewski", "birthYear": 1979},
]

print(f"Załadowano {len(SUSPECTS)} podejrzanych.")

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Odległość [km] między dwoma punktami GPS."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def get_locations(name, surname):
    """Pobierz listę lokalizacji GPS danej osoby."""
    resp = requests.post(f"{HUB_BASE}/api/location", json={
        "apikey": API_KEY, "name": name, "surname": surname
    })
    return resp.json()


def get_access_level(name, surname, birth_year):
    """Pobierz poziom dostępu danej osoby."""
    resp = requests.post(f"{HUB_BASE}/api/accesslevel", json={
        "apikey": API_KEY, "name": name, "surname": surname, "birthYear": birth_year
    })
    return resp.json()


def get_power_plants():
    """Pobierz listę elektrowni z ich kodami i współrzędnymi."""
    resp = requests.get(f"{HUB_BASE}/data/{API_KEY}/findhim_locations.json")
    return resp.json()


print("Funkcje pomocnicze gotowe.")

In [ ]:
# Definicje narzędzi dla Function Calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_person_locations",
            "description": "Pobierz listę lokalizacji GPS danej osoby z API.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name":    {"type": "string", "description": "Imię osoby"},
                    "surname": {"type": "string", "description": "Nazwisko osoby"}
                },
                "required": ["name", "surname"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_person_access_level",
            "description": "Pobierz poziom dostępu danej osoby.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name":      {"type": "string"},
                    "surname":   {"type": "string"},
                    "birthYear": {"type": "integer", "description": "Rok urodzenia"}
                },
                "required": ["name", "surname", "birthYear"]
            }
        }
    }
]

print("Narzędzia zdefiniowane.")

In [ ]:
# Pobierz elektrownie i przygotuj kontekst dla agenta
power_plants = get_power_plants()
print(f"Elektrownie: {json.dumps(power_plants, ensure_ascii=False, indent=2)[:500]}")

context = f"""
Lista podejrzanych (JSON):
{json.dumps(SUSPECTS, ensure_ascii=False)}

Lista elektrowni atomowych z kodami i współrzędnymi:
{json.dumps(power_plants, ensure_ascii=False)}

Twoje zadanie:
1. Dla każdej osoby pobierz jej lokalizacje GPS za pomocą narzędzia get_person_locations.
2. Oblicz odległość Haversine do każdej elektrowni (wzór: 2*R*asin(sqrt(sin^2(dlat/2) + cos(lat1)*cos(lat2)*sin^2(dlon/2))), R=6371 km).
3. Znajdź osobę, która była NAJBLIŻEJ JAKIEJKOLWIEK elektrowni (minimum ze wszystkich odległości).
4. Dla tej osoby pobierz poziom dostępu za pomocą narzędzia get_person_access_level.
5. Zwróć JSON: {{"name": ..., "surname": ..., "accessLevel": ..., "powerPlant": "KOD_ELEKTROWNI"}}
"""

messages = [
    {"role": "system", "content": "Jesteś agentem śledczym. Używaj dostępnych narzędzi do znalezienia podejrzanego przy elektrowni."},
    {"role": "user",   "content": context}
]

MAX_ITERS = 20
answer = None

for i in range(MAX_ITERS):
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    msg = resp.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        print(f"\nAgent odpowiedział (iter {i+1}): {msg.content}")
        try:
            # Wyciągnij JSON z odpowiedzi
            txt = msg.content
            start = txt.find("{")
            end   = txt.rfind("}") + 1
            answer = json.loads(txt[start:end])
        except Exception as e:
            print(f"Błąd parsowania: {e}")
        break

    # Wykonaj wywołania narzędzi
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        fn   = tc.function.name
        print(f"  → {fn}({args})")

        if fn == "get_person_locations":
            result = get_locations(args["name"], args["surname"])
        elif fn == "get_person_access_level":
            result = get_access_level(args["name"], args["surname"], args["birthYear"])
        else:
            result = {"error": "Nieznane narzędzie"}

        print(f"     wynik: {str(result)[:200]}")
        messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id})

print("\nKońcowa odpowiedź agenta:", answer)

In [ ]:
# Wyślij odpowiedź do Hub-u
assert answer, "Brak odpowiedzi agenta — sprawdź listę SUSPECTS"

payload = {"apikey": API_KEY, "task": "findhim", "answer": answer}
resp = requests.post(VERIFY_URL, json=payload)
print(resp.json())